# Shuffle Optimization: Partition Tuning, Bucket Joins, and Reducing I/O

**Module**: 10_performance_and_optimization  
**Notebook**: 04_shuffle_optimization.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Techniques to optimize shuffles: right-sizing shuffle partitions, bucketing to eliminate joins, and AQE partition coalescing.
- **Why does this topic exist?**: Shuffle optimization yields the highest ROI for big data jobs.
- **What problem does it solve?**: Default 200 shuffle partitions creating massive scheduling latency on small data, or spilling on large data.
- **Where is it used in real-world Data Engineering?**: Tuning heavy ETL jobs with multi-terabyte fact tables.

---

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Shuffle_Opt_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
# Tune shuffle partitions
spark.conf.set("spark.sql.shuffle.partitions", "8") # Match cluster capacity
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
res = orders.groupBy("channel").count()
res.show()
print("Shuffle executed with 8 partitions!")

---

## Experiment: AQE Automatic Partition Coalescing

- **1. What are we testing?**: Verify AQE coalesces 200 partitions into fewer partitions at runtime.
- **2. What do we expect?**: AQE merges small partitions automatically.

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "100") # Deliberately oversized
res2 = orders.groupBy("status").count()
res2.show()
print("AQE coalesced partitions automatically in runtime plan!")
spark.conf.set("spark.sql.shuffle.partitions", "8")

**3. What happened?**: AQE reduced 100 shuffle partitions to fewer active post-shuffle tasks.  
**4. Why did it happen?**: spark.sql.adaptive.coalescePartitions.enabled was active.  
**5. Practical Engineering Lesson**: AQE protects against oversized shuffle partition settings.

---

## Practical Exercises

### Exercise 1 (Beginner): Set Shuffle Partitions
**Problem Statement**: Set shuffle partitions to 16.  
**Input / Context**: spark.conf

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Bucketing for Zero Shuffle
**Problem Statement**: Explain bucketBy.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Shuffle Compression Codec
**Problem Statement**: Check spark.io.compression.codec.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Set Shuffle Partitions):
```python
spark.conf.set('spark.sql.shuffle.partitions','16')
```

#### Solution 2 (Bucketing for Zero Shuffle):
```python
print('Pre-partitions data by hash on disk; subsequent joins on that key require zero shuffle.')
```

#### Solution 3 (Shuffle Compression Codec):
```python
print(spark.conf.get('spark.io.compression.codec','lz4'))
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Sizing**: Target 100-200 MB per shuffle partition.
- **AQE**: Enables dynamic coalescing.

### Essential Technical Terms
- `shuffle.partitions`: Output partitions after wide transformation.
- `Bucketing`: Pre-sorted storage partitioning.

### Common Engineering Mistakes
- **Leaving default 200 on tiny data**: Creates 200 tiny tasks with massive scheduling overhead.

### Interview Questions on This Topic
- **Q: How does bucketing eliminate join shuffles?**
  - *A*: If two tables are bucketed on the same join key with the same number of buckets, corresponding buckets reside on the same partition index, allowing a narrow join without a shuffle.

### What to Study Next
Open 05_memory_optimization.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')